# Recurrent Depth Transformers: From Basic to Advanced

Standard Transformers stack `N` *unique* blocks. Recurrent-depth models instead **reuse the same block(s) along the depth axis**, turning depth into an iterative computation. This trades parameters for FLOPs, gives a natural knob for *test-time compute scaling*, and (for some tasks) lets a model reason in latent space rather than in tokens.

We will build, train and compare four architectures on a synthetic reasoning task (bit parity), in increasing order of sophistication:

1. **Stacked Transformer** — the usual baseline: `N` unique blocks.
2. **Weight-shared Recurrent Transformer** — *one* block applied `N` times. Same parameter count as a 1-layer model.
3. **Universal Transformer with ACT** — adds a learned halting mechanism so each token can choose its own depth (Dehghani et al., 2018).
4. **Latent Recurrent Depth (Huginn-style)** — a `prelude → recurrent core → coda` design that injects token embeddings into every iteration and uses a randomly-initialised latent state, enabling iteration counts that differ between train and test (Geiping et al., 2025).

Then we will look at two things you can only do with recurrent-depth models:
* **test-time depth scaling** — train with `K=4`, evaluate with `K=16`;
* **latent-state convergence** — watch the per-iteration delta `‖s_{k+1} − s_k‖` decay.

## 0. Setup

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. The shared building block: a pre-norm Transformer block

All four architectures use the same primitive: a pre-LayerNorm encoder block,

$$
\begin{aligned}
x' &= x + \text{Attn}(\text{LN}(x)) \\
y  &= x' + \text{MLP}(\text{LN}(x'))
\end{aligned}
$$

The pre-norm placement matters for recurrent depth: it keeps the residual stream's scale stable across many block applications, which is exactly the regime we are about to push into.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.dropout(h)
        h = self.norm2(x)
        return x + self.dropout(self.mlp(h))

## 2. Baseline: the standard stacked Transformer

A stack of `n_layers` *independent* blocks. Parameter count grows linearly with depth.

In [ ]:
class StackedTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, d_ff=128, n_layers=4, max_len=64, n_classes=2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0).expand(B, L)
        h = self.tok_emb(x) + self.pos_emb(pos)
        for blk in self.blocks:
            h = blk(h)
        return self.head(self.norm(h)[:, 0])  # read out at the [CLS] position

## 3. Weight-shared recurrent transformer

Replace the list of blocks with a *single* block that we apply `n_iters` times:

$$
h_{k+1} = f_\theta(h_k), \qquad k = 0, \dots, K-1.
$$

This is the simplest recurrent-depth model — sometimes called a "looped" or "tied-weight" Transformer. The benefits are:

* **parameter efficiency** — same params as a 1-layer model;
* **arbitrary depth at inference** — `n_iters` can change after training;
* **inductive bias toward iterative computation** — the same operator is applied at every step, so the model is encouraged to learn a *dynamics* rather than a layer-specific transform.

The cost is that it cannot specialise per-layer (no "early-layer = syntax / late-layer = semantics").

In [ ]:
class RecurrentDepthTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, d_ff=128, n_iters=4, max_len=64, n_classes=2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.block = TransformerBlock(d_model, n_heads, d_ff)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)
        self.n_iters = n_iters

    def forward(self, x, n_iters=None):
        n = n_iters if n_iters is not None else self.n_iters
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0).expand(B, L)
        h = self.tok_emb(x) + self.pos_emb(pos)
        for _ in range(n):
            h = self.block(h)
        return self.head(self.norm(h)[:, 0])

## 4. Universal Transformer with Adaptive Computation Time (ACT)

The Universal Transformer (Dehghani et al., 2018) takes the weight-shared idea and adds two ingredients:

1. **A depth/time embedding** added to the state at every iteration so the block knows *which* iteration it is on.
2. **ACT halting**: a small head per token outputs $p_n^{(t)} \in [0,1]$, and we accumulate halting probability until it crosses $1-\varepsilon$. The output is the halting-weighted sum of intermediate states, and a *ponder cost* $N + R$ (number of updates + remainder) is added to the loss to keep the model parsimonious.

This is the first model in the notebook where **different tokens use different amounts of compute**.

In [ ]:
class UniversalTransformerACT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, d_ff=128,
                 max_iters=8, halt_eps=0.01, max_len=64, n_classes=2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.iter_emb = nn.Embedding(max_iters, d_model)
        self.block = TransformerBlock(d_model, n_heads, d_ff)
        self.halt = nn.Linear(d_model, 1)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)
        self.max_iters = max_iters
        self.halt_eps = halt_eps
        self.last_ponder = torch.tensor(0.0)

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0).expand(B, L)
        state = self.tok_emb(x) + self.pos_emb(pos)

        halt_prob = torch.zeros(B, L, device=x.device)
        n_updates = torch.zeros(B, L, device=x.device)
        accum = torch.zeros_like(state)
        threshold = 1.0 - self.halt_eps

        for step in range(self.max_iters):
            still_running = (halt_prob < threshold).float()
            if still_running.sum() == 0:
                break
            sig = self.iter_emb(torch.tensor(step, device=x.device))
            state = self.block(state + sig)
            p = torch.sigmoid(self.halt(state)).squeeze(-1)

            new_halted = ((halt_prob + p) >= threshold).float() * still_running
            weight = (p * still_running * (1 - new_halted)
                      + (1 - halt_prob) * new_halted)
            accum = accum + weight.unsqueeze(-1) * state
            halt_prob = halt_prob + weight
            n_updates = n_updates + still_running

        # Ponder cost (N + R): used as auxiliary loss to discourage runaway compute.
        self.last_ponder = (n_updates + (1 - halt_prob)).mean()
        h = self.norm(accum)
        return self.head(h[:, 0])

## 5. Latent recurrent depth (Huginn-style)

Geiping et al. (2025) propose a more expressive variant tuned for *test-time reasoning*. The model has three pieces:

* a **prelude** $P$ of a few standard blocks that turn input tokens into embeddings $e$;
* a **recurrent core** $R$ that maintains a latent state $s$ and is iterated $K$ times;
* a **coda** $C$ that decodes the final latent into logits.

Crucially, at every iteration the embedding $e$ is *re-injected*:

$$
s_{k+1} = R\bigl(W_c\, [s_k \,\|\, e]\bigr),
$$

and the initial state $s_0$ is sampled from $\mathcal N(0, \sigma^2 I)$ rather than being deterministic. The random init + re-injection together encourage the recurrent core to behave like a contraction toward a fixed point, which is what makes *training with $K=4$, evaluating with $K=32$* actually work.

In [ ]:
class LatentRecurrentDepthTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, d_ff=128,
                 n_prelude=1, n_coda=1, n_iters=4, max_len=64, n_classes=2,
                 init_std=0.4):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.prelude = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_prelude)])
        self.core = TransformerBlock(d_model, n_heads, d_ff)
        self.coda = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_coda)])
        self.combine = nn.Linear(2 * d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)
        self.n_iters = n_iters
        self.init_std = init_std

    def forward(self, x, n_iters=None, return_trace=False):
        n = n_iters if n_iters is not None else self.n_iters
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0).expand(B, L)
        e = self.tok_emb(x) + self.pos_emb(pos)
        for blk in self.prelude:
            e = blk(e)

        s = torch.randn_like(e) * self.init_std
        trace = [s.detach()] if return_trace else None
        for _ in range(n):
            s = self.core(self.combine(torch.cat([s, e], dim=-1)))
            if return_trace:
                trace.append(s.detach())

        h = s
        for blk in self.coda:
            h = blk(h)
        h = self.norm(h)
        out = self.head(h[:, 0])
        return (out, trace) if return_trace else out

## 6. A toy reasoning task: bit parity

Parity (the XOR of all bits in the input) is the canonical example of a task where **depth matters**. A 1-layer Transformer with limited width has a hard time learning it, while iterated computation makes it easy: each iteration can refine a running XOR estimate.

Vocab: `0`, `1`, plus a `[CLS]` token at position 0 used as the readout.

In [ ]:
CLS = 2
VOCAB_SIZE = 3

def make_parity_dataset(n_samples: int, seq_len: int = 8, seed: int = 0):
    g = torch.Generator().manual_seed(seed)
    bits = torch.randint(0, 2, (n_samples, seq_len), generator=g)
    target = bits.sum(dim=1) % 2
    cls = torch.full((n_samples, 1), CLS, dtype=torch.long)
    return torch.cat([cls, bits], dim=1), target

SEQ_LEN = 8
X_train, y_train = make_parity_dataset(8000, SEQ_LEN, seed=1)
X_val, y_val     = make_parity_dataset(1000, SEQ_LEN, seed=2)
MAX_LEN = X_train.shape[1]
print('Train:', X_train.shape, 'class balance:', y_train.float().mean().item())
print('Sample:', X_train[0].tolist(), '->', y_train[0].item())

## 7. Parameter counts at matched depth

Notice how dramatically the parameter budget changes when you tie weights across depth.

In [ ]:
def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

configs = [
    ('Stacked (4 blocks)',          StackedTransformer(VOCAB_SIZE, n_layers=4, max_len=MAX_LEN)),
    ('Recurrent (1 block, K=4)',    RecurrentDepthTransformer(VOCAB_SIZE, n_iters=4, max_len=MAX_LEN)),
    ('Universal+ACT (max K=8)',     UniversalTransformerACT(VOCAB_SIZE, max_iters=8, max_len=MAX_LEN)),
    ('Latent recurrent (P+R+C, K=4)', LatentRecurrentDepthTransformer(VOCAB_SIZE, n_iters=4, max_len=MAX_LEN)),
]
for name, m in configs:
    print(f'{name:35s}  {n_params(m):>8,d} params')

## 8. Training loop

A single helper that trains any of the four models. For ACT we add the ponder cost as an auxiliary loss with a small weight.

In [ ]:
def train_model(model, X_tr, y_tr, X_va, y_va,
                epochs=15, batch_size=128, lr=3e-3, ponder_weight=0.0):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    history = {'train_loss': [], 'val_acc': []}
    X_va_d, y_va_d = X_va.to(device), y_va.to(device)
    for ep in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            if ponder_weight > 0 and hasattr(model, 'last_ponder'):
                loss = loss + ponder_weight * model.last_ponder
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            running += loss.item() * xb.size(0)
        running /= len(X_tr)
        model.eval()
        with torch.no_grad():
            acc = (model(X_va_d).argmax(-1) == y_va_d).float().mean().item()
        history['train_loss'].append(running)
        history['val_acc'].append(acc)
        print(f'  epoch {ep+1:2d}  loss={running:.4f}  val_acc={acc:.3f}')
    return history

## 9. Train the four models and compare

In [ ]:
EPOCHS = 15
histories = {}

print('Stacked Transformer (4 unique blocks)')
torch.manual_seed(0)
histories['Stacked'] = train_model(
    StackedTransformer(VOCAB_SIZE, n_layers=4, max_len=MAX_LEN),
    X_train, y_train, X_val, y_val, epochs=EPOCHS)

print('\nWeight-shared Recurrent (K=4)')
torch.manual_seed(0)
histories['Recurrent'] = train_model(
    RecurrentDepthTransformer(VOCAB_SIZE, n_iters=4, max_len=MAX_LEN),
    X_train, y_train, X_val, y_val, epochs=EPOCHS)

print('\nUniversal Transformer with ACT (max K=8)')
torch.manual_seed(0)
histories['ACT'] = train_model(
    UniversalTransformerACT(VOCAB_SIZE, max_iters=8, max_len=MAX_LEN),
    X_train, y_train, X_val, y_val, epochs=EPOCHS, ponder_weight=0.01)

print('\nLatent Recurrent Depth (Huginn-style, K=4)')
torch.manual_seed(0)
histories['Latent'] = train_model(
    LatentRecurrentDepthTransformer(VOCAB_SIZE, n_iters=4, max_len=MAX_LEN),
    X_train, y_train, X_val, y_val, epochs=EPOCHS)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, h in histories.items():
    ax[0].plot(h['train_loss'], label=name)
    ax[1].plot(h['val_acc'],   label=name)
ax[0].set(xlabel='epoch', ylabel='train loss', title='Training loss')
ax[1].set(xlabel='epoch', ylabel='val accuracy', title='Validation accuracy')
ax[0].legend(); ax[1].legend()
ax[0].grid(alpha=0.3); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Test-time depth scaling

Recurrent-depth models expose a knob — the number of iterations — that didn't exist in the stacked baseline. We trained the latent model with `K=4`; let's see what happens when we run it with `K ∈ {1, 2, 4, 8, 16, 24, 32}` at inference time.

In [ ]:
torch.manual_seed(0)
latent = LatentRecurrentDepthTransformer(VOCAB_SIZE, n_iters=4, max_len=MAX_LEN).to(device)
_ = train_model(latent, X_train, y_train, X_val, y_val, epochs=EPOCHS)

iters_to_try = [1, 2, 4, 8, 16, 24, 32]
X_va_d, y_va_d = X_val.to(device), y_val.to(device)
latent.eval()
accs = []
with torch.no_grad():
    for k in iters_to_try:
        # average over a few samples of s_0 to reduce randomness from the init
        preds = torch.stack([latent(X_va_d, n_iters=k).softmax(-1) for _ in range(4)]).mean(0)
        accs.append((preds.argmax(-1) == y_va_d).float().mean().item())

plt.figure(figsize=(6, 4))
plt.plot(iters_to_try, accs, marker='o')
plt.axvline(4, ls='--', color='k', alpha=0.5, label='train-time K')
plt.xlabel('inference iterations K'); plt.ylabel('val accuracy')
plt.title('Test-time depth scaling (latent recurrent model)')
plt.grid(alpha=0.3); plt.legend(); plt.show()

## 11. Latent-state convergence

If the recurrent core has learned a contractive update, the per-iteration change `‖s_{k+1} − s_k‖` should decay roughly geometrically — i.e. the latent is converging toward a fixed point that encodes the answer. This is the property that justifies the test-time scaling above.

In [ ]:
latent.eval()
with torch.no_grad():
    _, trace = latent(X_val[:128].to(device), n_iters=24, return_trace=True)

deltas = [(trace[i+1] - trace[i]).norm(dim=-1).mean().item()
          for i in range(len(trace) - 1)]

plt.figure(figsize=(6, 4))
plt.semilogy(range(1, len(trace)), deltas, marker='o')
plt.xlabel('iteration k'); plt.ylabel(r'$\Vert s_{k+1} - s_k \Vert$  (mean)')
plt.title('Latent state convergence')
plt.grid(alpha=0.3, which='both'); plt.show()

## 12. Where to go next

A few directions you can explore by editing the cells above:

* **Harder tasks.** Parity is intentionally tiny. Try modular addition chains (sum mod $p$ over a long sequence), associative recall, or sorting — these benefit even more from extra iterations.
* **Truncated backprop through depth.** With large `K`, backprop through every iteration becomes expensive. The Huginn paper backprops through only the last few iterations; you can do this by detaching `s` partway through the loop.
* **Deep Equilibrium Models (DEQ).** Instead of a fixed `K`, solve $s^* = R([s^* \,\|\, e])$ with a root-finder and use implicit differentiation for gradients. The recurrent-depth design above is essentially the unrolled, finite-step approximation.
* **Per-token vs per-sequence halting.** Our ACT halts per token. For classification you may want to halt per *sequence* — share the halting probability across positions.
* **KV-cache reuse across iterations.** In autoregressive settings, the embedding `e` is fixed during decoding; you can cache its keys/values and only recompute what depends on `s`.